# Pretrained ViT + LoRA Continual Learning on Split CIFAR-100

Clean debug notebook for the next supervisor update.

Main idea:

**ImageNet-pretrained ViT-B/16 → Split CIFAR-100 → LoRA per step → simple average merging**

Debug mode uses **1 epoch** first. After the notebook runs without errors, increase the epochs in Cell 5.

## Cell 1 — Imports

In [ ]:
import os
import json
import shutil
import random
import copy

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms
from datasets import load_dataset, concatenate_datasets, Image as HFImage

from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import set_seed

from peft import LoraConfig, get_peft_model

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Cell 2 — Run name and output folders

In [ ]:
RUN_NAME = "pretrained_vit_lora_simpleavg_debug"

OUTPUT_DIR = os.path.join("outputs", RUN_NAME)
RESULTS_DIR = os.path.join("results", RUN_NAME)
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")

for d in [OUTPUT_DIR, RESULTS_DIR, TABLES_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## Cell 3 — Load CIFAR-100

In [ ]:
dataset = load_dataset("cifar100")
dataset = dataset.rename_column("fine_label", "label")
dataset = dataset.cast_column("img", HFImage())

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

print("num_classes:", num_classes)
print("train size:", len(dataset["train"]))
print("test size:", len(dataset["test"]))
print("first 10 classes:", class_names[:10])

## Cell 4 — Split CIFAR-100 into 5 steps

Default debug split: **5 steps × 20 classes**.

In [ ]:
NUM_STEPS = 5
assert num_classes % NUM_STEPS == 0

CLASSES_PER_STEP = num_classes // NUM_STEPS
class_splits = [
    list(range(step * CLASSES_PER_STEP, (step + 1) * CLASSES_PER_STEP))
    for step in range(NUM_STEPS)
]

first_step_classes = class_splits[0]
later_step_classes = [c for step in range(1, NUM_STEPS) for c in class_splits[step]]
all_classes = list(range(num_classes))

print("classes per step:", CLASSES_PER_STEP)
for i, cls in enumerate(class_splits, start=1):
    print(f"step {i}: {cls[0]}-{cls[-1]} ({len(cls)} classes)")

## Cell 5 — Debug hyperparameters

Start with **1 epoch** for all methods. Increase later only after the notebook is clean.

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEBUG_MODE = True

# Start with 1 epoch for debugging.
FT_EPOCHS = 1
LORA_EPOCHS = 1
JOINT_EPOCHS = 1
LORAC_IPC_EPOCHS = 1

BATCH_FT = 8
ACCUM_FT = 2

BATCH_LORA = 16
ACCUM_LORA = 1

LR_FT = 3e-5
LR_LORA = 5e-5
LR_JOINT = 5e-5
LR_LORAC_IPC = 5e-5

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"
USE_FP16 = torch.cuda.is_available()

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["query", "value"]

# Small replay diagnostic beside simple_avg_no_replay.
REPLAY_PER_CLASS = 20

# Paper-inspired variants.
LORAC_IPC_LAMBDA_ORTH = 0.1

METHODS_TO_RUN = {
    "seq_ft_no_replay": True,
    "lora_no_replay": True,
    "simple_avg_no_replay": True,
    "simple_avg_replay": True,
    "do_merging": True,
    "lorac_ipc": True,
    "joint_upper_bound": True,
}

print(json.dumps(METHODS_TO_RUN, indent=2))

## Cell 6 — Model and preprocessing

In [ ]:
MODEL_CHECKPOINT = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

size = image_processor.size
if isinstance(size, dict):
    H = int(size.get("height", 224))
    W = int(size.get("width", 224))
else:
    H = W = int(size)

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_processor.image_mean, std=image_processor.image_std),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")
    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)
    if isinstance(x, np.ndarray):
        arr = np.squeeze(x).astype(np.uint8)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))
        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        return Image.fromarray(arr).convert("RGB")
    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex["img"]]
    ex["labels"] = ex["label"]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": float((preds == labels).mean())}

def fresh_pretrained_model():
    config = AutoConfig.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=num_classes,
        id2label={i: str(i) for i in range(num_classes)},
        label2id={str(i): i for i in range(num_classes)},
    )
    return AutoModelForImageClassification.from_pretrained(
        MODEL_CHECKPOINT,
        config=config,
        ignore_mismatched_sizes=True,
    )

def add_lora(model):
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        modules_to_save=["classifier"],
    )
    return get_peft_model(model, lora_config)

## Cell 7 — Dataset helpers with optional small replay

In [ ]:
def filter_by_classes(ds, class_ids):
    class_set = set(int(c) for c in class_ids)
    ds = ds.with_format(None)
    return ds.filter(lambda ex: int(ex["label"]) in class_set)

def classes_until_step(step_idx):
    out = []
    for s in range(step_idx + 1):
        out.extend(class_splits[s])
    return out

def make_eval_dataset(class_ids):
    ds = filter_by_classes(dataset["test"], class_ids)
    ds = ds.with_transform(preprocess_val)
    return ds

def make_new_train_dataset(step_idx):
    new_classes = class_splits[step_idx]
    ds = filter_by_classes(dataset["train"], new_classes)
    ds = ds.with_transform(preprocess_train)
    return ds

def make_replay_subset(old_classes, replay_per_class):
    if replay_per_class <= 0 or len(old_classes) == 0:
        return None

    replay_parts = []
    for c in old_classes:
        class_ds = filter_by_classes(dataset["train"], [c]).shuffle(seed=SEED + int(c))
        n = min(replay_per_class, len(class_ds))
        replay_parts.append(class_ds.select(range(n)))

    if not replay_parts:
        return None
    return concatenate_datasets(replay_parts)

def make_train_dataset(step_idx, replay_per_class=0):
    new_classes = class_splits[step_idx]
    old_classes = [c for s in range(step_idx) for c in class_splits[s]]

    new_ds = filter_by_classes(dataset["train"], new_classes)
    replay_ds = make_replay_subset(old_classes, replay_per_class)

    if replay_ds is None:
        train_ds = new_ds
    else:
        train_ds = concatenate_datasets([new_ds, replay_ds]).shuffle(seed=SEED + step_idx)

    train_ds = train_ds.with_transform(preprocess_train)

    print(f"[train dataset] step={step_idx+1} new_classes={len(new_classes)} old_classes={len(old_classes)} replay_per_class={replay_per_class} size={len(train_ds)}")
    return train_ds

EVAL_FIRST = make_eval_dataset(first_step_classes)
EVAL_LATER = make_eval_dataset(later_step_classes)
EVAL_ALL = make_eval_dataset(all_classes)

print("eval first:", len(EVAL_FIRST))
print("eval later:", len(EVAL_LATER))
print("eval all:", len(EVAL_ALL))

## Cell 8 — Generic training and evaluation helpers

In [ ]:
all_results = []

def make_training_args(output_dir, epochs, lr, batch_size, accum_steps, eval_dataset_present=True):
    return TrainingArguments(
        output_dir=output_dir,
        remove_unused_columns=False,
        eval_strategy="epoch" if eval_dataset_present else "no",
        save_strategy="no",
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=accum_steps,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        report_to="none",
        max_grad_norm=1.0,
    )

def train_with_trainer(model, train_ds, eval_ds, output_dir, epochs, lr, batch_size, accum_steps, trainer_cls=Trainer, **trainer_kwargs):
    args = make_training_args(
        output_dir=output_dir,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        accum_steps=accum_steps,
        eval_dataset_present=eval_ds is not None,
    )
    trainer = trainer_cls(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        **trainer_kwargs,
    )
    trainer.train()
    return trainer

def evaluate_model(model, method_name, step=NUM_STEPS):
    args = TrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, f"eval_{method_name}"),
        remove_unused_columns=False,
        report_to="none",
        fp16=USE_FP16,
        per_device_eval_batch_size=32,
    )
    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    evals = {
        "first_step": trainer.evaluate(EVAL_FIRST),
        "later_steps": trainer.evaluate(EVAL_LATER),
        "all_seen": trainer.evaluate(EVAL_ALL),
    }

    for eval_set, metrics in evals.items():
        all_results.append({
            "method": method_name,
            "step": step,
            "eval_set": eval_set,
            "accuracy": float(metrics.get("eval_accuracy", np.nan)),
            "loss": float(metrics.get("eval_loss", np.nan)),
        })

    print(method_name, {k: round(v["eval_accuracy"], 4) for k, v in evals.items()})
    return evals

## Cell 9 — LoRA extraction, merging, and classifier stitching

In [ ]:
def normalize_module_name(name):
    for prefix in ["base_model.model.", "model."]:
        if name.startswith(prefix):
            name = name[len(prefix):]
    return name

def get_submodule(model, name):
    module = model
    for part in name.split("."):
        module = getattr(module, part)
    return module

def get_classifier_tensors(model):
    root = model.base_model.model if hasattr(model, "base_model") else model
    clf = root.classifier
    if hasattr(clf, "modules_to_save") and "default" in clf.modules_to_save:
        clf = clf.modules_to_save["default"]
    weight = clf.weight.detach().cpu().clone()
    bias = clf.bias.detach().cpu().clone() if clf.bias is not None else None
    return weight, bias

def extract_lora_state(model):
    deltas = {}
    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )
        if not has_lora:
            continue

        A = module.lora_A["default"].weight.detach().cpu()
        B = module.lora_B["default"].weight.detach().cpu()
        scaling = module.scaling["default"] if isinstance(module.scaling, dict) else module.scaling
        delta = (B @ A) * float(scaling)

        # The corresponding plain ViT module name.
        plain_name = normalize_module_name(name)
        deltas[plain_name] = delta.clone()

    clf_w, clf_b = get_classifier_tensors(model)
    return {"deltas": deltas, "classifier_weight": clf_w, "classifier_bias": clf_b}

def stitch_classifier(base_model, step_states):
    with torch.no_grad():
        for step_idx, state in enumerate(step_states):
            rows = class_splits[step_idx]
            base_model.classifier.weight[rows, :] = state["classifier_weight"][rows, :].to(base_model.classifier.weight.device)
            if base_model.classifier.bias is not None and state["classifier_bias"] is not None:
                base_model.classifier.bias[rows] = state["classifier_bias"][rows].to(base_model.classifier.bias.device)
    return base_model

def apply_deltas_to_base(delta_dict, step_states_for_classifier):
    model = fresh_pretrained_model()
    model.eval()
    with torch.no_grad():
        for module_name, delta in delta_dict.items():
            target = get_submodule(model, module_name)
            target.weight += delta.to(target.weight.device, dtype=target.weight.dtype)
    model = stitch_classifier(model, step_states_for_classifier)
    return model

def simple_average_deltas(step_states):
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}
    for k in keys:
        merged[k] = torch.stack([s["deltas"][k] for s in step_states], dim=0).mean(dim=0)
    return merged

def do_merge_deltas(step_states, eps=1e-8):
    """
    Minimal data-free DO-style merge for debugging:
    1) decouple each full-rank LoRA delta into column magnitude and direction;
    2) average magnitudes;
    3) average normalized directions.

    This keeps the notebook simple. It is paper-inspired, not a full reproduction of the paper's
    layer-wise optimization procedure.
    """
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    for k in keys:
        matrices = [s["deltas"][k].float() for s in step_states]
        mags = []
        dirs = []

        for W_delta in matrices:
            col_mag = torch.linalg.norm(W_delta, dim=0, keepdim=True).clamp_min(eps)
            mags.append(col_mag)
            dirs.append(W_delta / col_mag)

        mag_merge = torch.stack(mags, dim=0).mean(dim=0)
        dir_merge = torch.stack(dirs, dim=0).mean(dim=0)
        dir_merge = dir_merge / torch.linalg.norm(dir_merge, dim=0, keepdim=True).clamp_min(eps)
        merged[k] = (dir_merge * mag_merge).to(matrices[0].dtype)

    return merged

## Cell 10 — Baseline 1: `seq_ft_no_replay`

Sequential full fine-tuning on new classes only. This is the forgetting baseline.

In [ ]:
if METHODS_TO_RUN["seq_ft_no_replay"]:
    seq_ft_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(class_splits[step_idx])
        out_dir = os.path.join(OUTPUT_DIR, f"seq_ft_no_replay_step_{step_idx+1}")

        print(f"\n===== seq_ft_no_replay | step {step_idx+1}/{NUM_STEPS} =====")
        train_with_trainer(
            model=seq_ft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=FT_EPOCHS,
            lr=LR_FT,
            batch_size=BATCH_FT,
            accum_steps=ACCUM_FT,
        )

    evaluate_model(seq_ft_model, "seq_ft_no_replay")
else:
    print("Skipping seq_ft_no_replay")

## Cell 11 — Baseline 2: `lora_no_replay`

One LoRA adapter trained sequentially across steps, without replay.

In [ ]:
if METHODS_TO_RUN["lora_no_replay"]:
    lora_seq_model = add_lora(fresh_pretrained_model())
    lora_seq_model.print_trainable_parameters()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(class_splits[step_idx])
        out_dir = os.path.join(OUTPUT_DIR, f"lora_no_replay_step_{step_idx+1}")

        print(f"\n===== lora_no_replay | step {step_idx+1}/{NUM_STEPS} =====")
        train_with_trainer(
            model=lora_seq_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=LORA_EPOCHS,
            lr=LR_LORA,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
        )

    evaluate_model(lora_seq_model, "lora_no_replay")
else:
    print("Skipping lora_no_replay")

## Cell 12 — Train independent LoRAs without replay

These step LoRAs are used for:

- `simple_avg_no_replay`
- `do_merging`

In [ ]:
def train_independent_loras(method_prefix, replay_per_class=0, trainer_cls=Trainer, trainer_extra_kwargs_fn=None, epochs=LORA_EPOCHS, lr=LR_LORA):
    step_states = []

    for step_idx in range(NUM_STEPS):
        model = add_lora(fresh_pretrained_model())
        train_ds = make_train_dataset(step_idx, replay_per_class=replay_per_class)
        eval_ds = make_eval_dataset(class_splits[step_idx])
        out_dir = os.path.join(OUTPUT_DIR, f"{method_prefix}_step_{step_idx+1}")

        extra_kwargs = {}
        if trainer_extra_kwargs_fn is not None:
            extra_kwargs = trainer_extra_kwargs_fn(step_idx, step_states)

        print(f"\n===== {method_prefix} | step {step_idx+1}/{NUM_STEPS} | replay_per_class={replay_per_class} =====")
        train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=epochs,
            lr=lr,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=trainer_cls,
            **extra_kwargs,
        )

        step_states.append(extract_lora_state(model))
        del model
        torch.cuda.empty_cache()

    return step_states

step_states_no_replay = None

if METHODS_TO_RUN["simple_avg_no_replay"] or METHODS_TO_RUN["do_merging"]:
    step_states_no_replay = train_independent_loras(
        method_prefix="independent_lora_no_replay",
        replay_per_class=0,
    )
else:
    print("Skipping independent no-replay LoRAs")

## Cell 13 — Method 3: `simple_avg_no_replay`

Simple average of the step LoRA deltas, with classifier rows stitched from the step where each class first appears.

In [ ]:
if METHODS_TO_RUN["simple_avg_no_replay"]:
    assert step_states_no_replay is not None
    avg_delta = simple_average_deltas(step_states_no_replay)
    simple_avg_model = apply_deltas_to_base(avg_delta, step_states_no_replay)
    evaluate_model(simple_avg_model, "simple_avg_no_replay")
    del simple_avg_model
    torch.cuda.empty_cache()
else:
    print("Skipping simple_avg_no_replay")

## Cell 14 — Method 4: `simple_avg_replay`

Same simple average merging, but each step LoRA is trained with a small replay buffer from old classes.

In [ ]:
step_states_replay = None

if METHODS_TO_RUN["simple_avg_replay"]:
    step_states_replay = train_independent_loras(
        method_prefix="independent_lora_replay",
        replay_per_class=REPLAY_PER_CLASS,
    )

    avg_delta_replay = simple_average_deltas(step_states_replay)
    simple_avg_replay_model = apply_deltas_to_base(avg_delta_replay, step_states_replay)
    evaluate_model(simple_avg_replay_model, "simple_avg_replay")
    del simple_avg_replay_model
    torch.cuda.empty_cache()
else:
    print("Skipping simple_avg_replay")

## Cell 15 — Paper-inspired method 1: `do_merging`

Data-free LoRA merging inspired by **Decouple and Orthogonalize**.

For debugging, this cell uses the simple decoupling part: column magnitude + direction.

In [ ]:
if METHODS_TO_RUN["do_merging"]:
    assert step_states_no_replay is not None
    do_delta = do_merge_deltas(step_states_no_replay)
    do_model = apply_deltas_to_base(do_delta, step_states_no_replay)
    evaluate_model(do_model, "do_merging")
    del do_model
    torch.cuda.empty_cache()
else:
    print("Skipping do_merging")

## Cell 16 — Paper-inspired method 2: `lorac_ipc`

Minimal LoRAC-IPC-inspired debug variant:

- train one LoRA per step from the same pretrained ViT;
- add an orthogonality penalty against previous LoRA deltas;
- merge the resulting LoRAs by simple average.

This is kept simple for debugging before attempting a full paper reproduction.

In [ ]:
def compute_lora_orthogonal_penalty(model, previous_states, eps=1e-8):
    if not previous_states:
        return torch.tensor(0.0, device=next(model.parameters()).device)

    penalties = []

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )
        if not has_lora:
            continue

        plain_name = normalize_module_name(name)
        if plain_name not in previous_states[0]["deltas"]:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        scaling = module.scaling["default"] if isinstance(module.scaling, dict) else module.scaling
        current_delta = (B @ A) * float(scaling)
        current_vec = current_delta.flatten()
        current_norm = torch.linalg.norm(current_vec).clamp_min(eps)

        for state in previous_states:
            old_delta = state["deltas"][plain_name].to(current_delta.device, dtype=current_delta.dtype)
            old_vec = old_delta.flatten()
            old_norm = torch.linalg.norm(old_vec).clamp_min(eps)
            cos = torch.dot(current_vec, old_vec) / (current_norm * old_norm)
            penalties.append(cos.pow(2))

    if not penalties:
        return torch.tensor(0.0, device=next(model.parameters()).device)
    return torch.stack(penalties).mean()

class LoRACIPCTrainer(Trainer):
    def __init__(self, *args, previous_states=None, lambda_orth=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.previous_states = previous_states or []
        self.lambda_orth = float(lambda_orth)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss
        orth_loss = compute_lora_orthogonal_penalty(model, self.previous_states)
        loss = ce_loss + self.lambda_orth * orth_loss
        return (loss, outputs) if return_outputs else loss

def lorac_ipc_extra_kwargs(step_idx, previous_states):
    return {
        "previous_states": previous_states,
        "lambda_orth": LORAC_IPC_LAMBDA_ORTH,
    }

step_states_lorac_ipc = None

if METHODS_TO_RUN["lorac_ipc"]:
    step_states_lorac_ipc = train_independent_loras(
        method_prefix="lorac_ipc",
        replay_per_class=0,
        trainer_cls=LoRACIPCTrainer,
        trainer_extra_kwargs_fn=lorac_ipc_extra_kwargs,
        epochs=LORAC_IPC_EPOCHS,
        lr=LR_LORAC_IPC,
    )

    lorac_delta = simple_average_deltas(step_states_lorac_ipc)
    lorac_model = apply_deltas_to_base(lorac_delta, step_states_lorac_ipc)
    evaluate_model(lorac_model, "lorac_ipc")
    del lorac_model
    torch.cuda.empty_cache()
else:
    print("Skipping lorac_ipc")

## Cell 17 — Upper bound: `joint_upper_bound`

Joint training on all CIFAR-100 classes. This is the reference ceiling.

In [ ]:
if METHODS_TO_RUN["joint_upper_bound"]:
    joint_model = fresh_pretrained_model()
    joint_train = filter_by_classes(dataset["train"], all_classes).with_transform(preprocess_train)
    joint_eval = make_eval_dataset(all_classes)

    print("\n===== joint_upper_bound =====")
    train_with_trainer(
        model=joint_model,
        train_ds=joint_train,
        eval_ds=joint_eval,
        output_dir=os.path.join(OUTPUT_DIR, "joint_upper_bound"),
        epochs=JOINT_EPOCHS,
        lr=LR_JOINT,
        batch_size=BATCH_FT,
        accum_steps=ACCUM_FT,
    )

    evaluate_model(joint_model, "joint_upper_bound")
else:
    print("Skipping joint_upper_bound")

## Cell 18 — Final table and plots

In [ ]:
results_df = pd.DataFrame(all_results)
results_path = os.path.join(TABLES_DIR, "all_results.csv")
results_df.to_csv(results_path, index=False)

print("saved:", results_path)
print(results_df)

method_order = [
    "seq_ft_no_replay",
    "lora_no_replay",
    "simple_avg_no_replay",
    "simple_avg_replay",
    "do_merging",
    "lorac_ipc",
    "joint_upper_bound",
]

final_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="accuracy",
    aggfunc="mean",
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in final_table.columns:
        final_table[col] = np.nan

final_table = final_table.reindex([m for m in method_order if m in final_table.index])
final_table = final_table[["first_step", "later_steps", "all_seen"]]
final_table_percent = final_table * 100

final_table_path = os.path.join(TABLES_DIR, "final_accuracy_percent.csv")
final_table_percent.to_csv(final_table_path)

print("\nFinal accuracy (%):")
print(final_table_percent.round(2))

plt.figure(figsize=(10, 5))
plot_df = final_table_percent.reset_index()
x = np.arange(len(plot_df))
width = 0.25

plt.bar(x - width, plot_df["first_step"], width, label="first_step")
plt.bar(x, plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, plot_df["all_seen"], width, label="all_seen")
plt.xticks(x, plot_df["method"], rotation=30, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("Pretrained ViT + Split CIFAR-100: final comparison")
plt.legend()
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, "final_accuracy_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()

print("saved:", final_table_path)
print("saved:", plot_path)